# 07 · Advanced RAG: query rewriting, multi-query, HyDE, decomposition and agentic loops

After this notebook you can diagnose *why* a question retrieves badly and pick the matching fix: rewrite or expand the query, fan out into several queries and fuse them, search with a hypothetical answer (HyDE), step back to a broader question, decompose multi-part and multi-hop questions, compress the context, retrieve small and return big, filter on metadata, and run a bounded retrieve-judge-requery loop - each implemented and measured on the same labelled questions.

**Time:** ~40 min · **Runs:** offline on CPU in ~1 min (≈ 20 LLM calls, answered by the offline stand-in unless you set a key) · **Needs:** [RAG pipeline from scratch](06_rag_pipeline_from_scratch.ipynb)

## Why this matters in interviews

- "Your RAG misses questions users phrase differently - what do you do?" The strong answer names the **failure first** (vocabulary mismatch, multi-part, multi-hop, too vague, wrong scope) and then the technique for it.
- Every technique here costs an LLM call and latency. Interviewers listen for **when not to** use it: rewrite selectively, fan out only when retrieval looks weak, cap agent loops.
- Honesty about mechanics: HyDE works because answers look like answers; multi-query works because one embedding of one phrasing has a recall ceiling; agentic RAG earns its cost only on multi-hop questions.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `rg09` | What is query rewriting and what forms does it take? |
| `rg35` | What is multi-query / fan-out retrieval and when does it pay? |
| `rg26` | What is HyDE and why does it work? |
| `rg36` | How do you handle multi-hop questions in RAG? |
| `rg16` | What is agentic RAG, and how does it differ from classic RAG? |
| `rg10` | What is metadata filtering and why is it more important than it sounds? |
| `rg05` | What is parent-child (small-to-big) retrieval? |
| `rg17` | What is GraphRAG and when is it worth the indexing cost? |
| `rg15` | How do you optimise retrieval quality? Give me the ordered list. |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

np.random.seed(0)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)

In [ ]:
# The knowledge base: 15 short markdown docs for a fictional company, Acme Cloud (a SaaS analytics product
# that also sells an 'Edge Gateway' device). This cell is data - it is identical in every 09_rag notebook.
KB = {
"refund_policy.md": """# Refund Policy

## Eligibility
Annual plans can be refunded in full within 30 days of purchase. Monthly plans can be cancelled at any time, but the current month is not refunded. Add-on credits and professional-services hours are non-refundable.

## How to request a refund
Open the billing portal, choose the invoice and click "Request refund". Only workspace owners can request refunds.

## Processing time
Approved refunds go back to the original payment method within 5-10 business days. Bank transfers can take up to 15 business days.
""",
"shipping.md": """# Hardware Shipping

## Delivery times
Edge Gateway devices ship from our warehouses in Rotterdam and Dallas. Standard delivery takes 3-5 business days. Express delivery takes 1-2 business days and costs $25.

## Tracking
A tracking link is emailed when the order leaves the warehouse. The first carrier scan can take 24 hours to appear.

## Lost or damaged shipments
If a shipment has not arrived 10 business days after dispatch, report it within 14 days so we can open a carrier investigation. We send a free replacement when the carrier confirms the loss.
""",
"warranty.md": """# Edge Gateway Warranty

## Coverage
Every Edge Gateway has a 2-year limited hardware warranty from the delivery date. It covers manufacturing defects in the device, the power supply and the antennas.

## Not covered
Water damage, drops, unauthorised repairs and cosmetic scratches are not covered by the warranty.

## RMA process
Request an RMA (return merchandise authorisation) in the support portal with the device serial number. Once the RMA is approved we ship a replacement within 7 days, and you send the faulty unit back with the prepaid label.
""",
"api_rate_limits.md": """# API Rate Limits

## Limits per plan
Free: 60 requests per minute. Pro: 600 requests per minute. Business: 1,200 requests per minute. Enterprise: 6,000 requests per minute. Limits apply per API key.

## Best practices
Retry with exponential backoff and jitter, cache responses you read often, and send writes through the bulk endpoint.
""",
"api_errors.md": """# API Error Reference

## ERR-429-A
The per-minute rate limit was exceeded. Wait for the number of seconds in the Retry-After header, then retry.

## ERR-429-B
The daily request quota is exhausted. The quota resets at 00:00 UTC; upgrade the plan for a higher quota.

## ERR-503-C
The API is in scheduled maintenance. Retry after the maintenance window shown on the status page.

## ERR-401-D
The API key was revoked or has expired. Create a new key in Settings > API keys.
""",
"gateway_models.md": """# Edge Gateway Models

## EG-100
The EG-100 is the entry model: 2 Ethernet ports, Wi-Fi 5 and an 8 W maximum power draw. It supports up to 50 sensors.

## EG-200
The EG-200 adds LTE backup: 4 Ethernet ports, Wi-Fi 6 and a 12 W maximum power draw. It supports up to 200 sensors.

## EG-300
The EG-300 is the rugged outdoor model: an IP67 enclosure, 4 Ethernet ports, Wi-Fi 6 and an 18 W maximum power draw. It supports up to 500 sensors.
""",
"oncall_policy.md": """# On-call Policy

## Rotation
Engineers join the on-call rotation after their third month. Each shift lasts one week and hands over on Monday at 09:00 local time. Every shift has a primary and a secondary.

## Response targets
A SEV1 page must be acknowledged within 5 minutes and a SEV2 page within 30 minutes. If a SEV1 page is not acknowledged within 15 minutes, it escalates to the secondary and then to the engineering manager.

## Compensation
On-call engineers receive a stipend of $300 per week, and a day off after any night with more than two pages.
""",
"pricing.md": """# Pricing Plans

## Plans
Free costs $0 and includes 3 users and 1 GB of storage. Pro costs $49 per user per month. Business costs $99 per user per month. Enterprise pricing is custom and starts at 50 users.

## What each plan includes
Single sign-on (SAML and OIDC) is included from the Business plan upward. Audit logs and the 99.95% uptime SLA are included only in Enterprise. All paid plans include email support.

## Billing
Paying annually saves 20% compared with monthly billing. Prices exclude VAT.
""",
"sso_setup.md": """# Single Sign-On Setup

## Supported identity providers
Acme Cloud supports Okta, Microsoft Entra ID (formerly Azure AD) and Google Workspace through SAML 2.0 or OIDC.

## Configuration steps
Go to Settings > Security > Single sign-on, upload the identity provider's metadata XML, and map the email and groups attributes. Test with one account before you enforce SSO for the whole workspace.

## Troubleshooting
SSO-ERR-12 means the SAML assertion was rejected because the clocks differ by more than 5 minutes; sync the identity provider's clock with NTP. SSO-ERR-31 means the email attribute is missing from the assertion.
""",
"data_retention.md": """# Data Retention

## Event data
Raw event data is kept for 30 days on Free, 13 months on Pro and Business, and 25 months on Enterprise. Aggregated reports are kept for as long as the workspace exists.

## Deleted workspaces
When a workspace is deleted, its data is purged from primary storage after 30 days and from backups after a further 35 days.

## Erasure requests
GDPR erasure requests for one person's data are completed within 30 days of identity verification.
""",
"pto_policy.md": """# Paid Time Off

## Allowance
Full-time employees receive 25 days of paid time off per calendar year, plus public holidays. Part-time employees receive a pro-rata allowance.

## Booking
Book absences longer than three days in Workday at least two weeks ahead. Your manager approves the request.

## Carry-over
Up to 5 unused days can be carried into the next year and must be used by 31 March.
""",
"webhooks.md": """# Webhooks

## Delivery and retries
Webhook events are sent as HTTPS POST requests. If your endpoint does not return a 2xx status within 10 seconds, we retry up to 8 times with exponential backoff over 24 hours.

## Verifying signatures
Every request carries an X-Acme-Signature header: an HMAC-SHA256 of the raw body computed with your webhook secret. Compare it in constant time and reject requests older than 5 minutes.
""",
"sla.md": """# Service Level Agreement

## Uptime commitment
Pro and Business are backed by a 99.9% monthly uptime commitment. Enterprise is backed by 99.95%.

## Service credits
If monthly uptime falls below the commitment, you receive a service credit of 10% of the monthly fee; below 99.0% the credit is 25%. Credits are applied to the next invoice.

## How to claim
Open a support ticket within 30 days of the end of the affected month and include the incident dates.
""",
"account_security.md": """# Account Security

## Passwords
Password reset links expire after 60 minutes. After 5 failed sign-in attempts the account is locked for 15 minutes.

## Two-factor authentication
Workspace owners can require two-factor authentication (2FA) for every member. Authenticator apps and hardware security keys are supported; SMS codes are not.
""",
"expenses.md": """# Travel and Expenses

## Meals and hotels
Meals are reimbursed up to $75 per day while travelling. Book hotels through the travel portal; the nightly cap is $200 in most cities.

## Submitting claims
Submit receipts in Workday within 30 days of the expense. Claims over $25 without a receipt are rejected.
""",
}

In [ ]:
# The eval set: each question lists its EVIDENCE - (doc, phrase) pairs that a retrieved chunk must contain.
# Phrases are unique in the KB, so "is the evidence in the retrieved text?" works for any chunking scheme.
QUESTIONS = [
    {"q": "What does ERR-429-A mean?", "kind": "exact id", "evidence": [("api_errors.md", "per-minute rate limit")]},
    {"q": "Our integration keeps failing with ERR-401-D", "kind": "exact id", "evidence": [("api_errors.md", "revoked")]},
    {"q": "How do I fix SSO-ERR-12?", "kind": "exact id", "evidence": [("sso_setup.md", "clocks differ")]},
    {"q": "How many sensors does the EG-300 support?", "kind": "exact id", "evidence": [("gateway_models.md", "500 sensors")]},
    {"q": "How do I get my money back after buying an annual plan by mistake?", "kind": "paraphrase",
     "evidence": [("refund_policy.md", "refunded in full within 30 days")]},
    {"q": "My parcel never showed up. What should I do?", "kind": "paraphrase", "evidence": [("shipping.md", "report it within 14 days")]},
    {"q": "How many vacation days do I get a year?", "kind": "paraphrase", "evidence": [("pto_policy.md", "25 days of paid time off")]},
    {"q": "Do we get compensated for outages?", "kind": "paraphrase", "evidence": [("sla.md", "service credit")]},
    {"q": "Which gateway can be installed outdoors?", "kind": "paraphrase", "evidence": [("gateway_models.md", "rugged outdoor model")]},
    {"q": "Who gets paged when nobody acknowledges a SEV1 alert?", "kind": "keyword", "evidence": [("oncall_policy.md", "engineering manager")]},
    {"q": "How long is event data kept on the Pro plan?", "kind": "keyword", "evidence": [("data_retention.md", "13 months on Pro")]},
    {"q": "What do we get back if the platform is down more than promised?", "kind": "vague", "evidence": [("sla.md", "service credit")]},
    {"q": "What is the API rate limit on the cheapest plan that includes SSO?", "kind": "multi-hop",
     "evidence": [("pricing.md", "from the Business plan upward"), ("api_rate_limits.md", "Business: 1,200 requests")]},
    {"q": "How long do approved refunds take, and how long is the gateway warranty?", "kind": "multi-part",
     "evidence": [("refund_policy.md", "5-10 business days"), ("warranty.md", "2-year")]},
    {"q": "What is the capital of Australia?", "kind": "out-of-scope", "evidence": []},
]
IN_SCOPE = [q for q in QUESTIONS if q["evidence"]]


def has_evidence(text, doc, q):
    """True if this text (from `doc`) contains any of the question's evidence phrases."""
    return any(doc == d and p.lower() in text.lower() for d, p in q["evidence"])


print(f"{len(KB)} docs, {sum(len(t) for t in KB.values()):,} characters; "
      f"{len(QUESTIONS)} questions ({len(IN_SCOPE)} answerable, {len(QUESTIONS) - len(IN_SCOPE)} out of scope)")

In [ ]:
import re


def split_sections(kb):
    """One chunk per '## ' section. The heading path is prepended so every chunk says where it came from."""
    chunks = []
    for doc, text in kb.items():
        title = text.splitlines()[0].lstrip("# ").strip()
        for i, part in enumerate(re.split(r"^## ", text, flags=re.M)[1:]):
            heading, _, body = part.partition("\n")
            path = f"{title} > {heading.strip()}"
            chunks.append({"id": f"{doc}#{i}", "doc": doc, "section": path, "text": f"{path}\n{body.strip()}"})
    return chunks


def is_relevant(chunk, q):
    return has_evidence(chunk["text"], chunk["doc"], q)


CHUNKS = split_sections(KB)
print(f"{len(CHUNKS)} section chunks, avg {sum(len(c['text']) for c in CHUNKS) / len(CHUNKS):.0f} characters")
print("example:", CHUNKS[1]["id"], "->", CHUNKS[1]["text"][:90].replace("\n", " | "), "...")

In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import transformers
transformers.utils.logging.disable_progress_bar()          # keep the cell output clean
transformers.logging.set_verbosity_error()
from sentence_transformers import SentenceTransformer


def load_st(name, cls=SentenceTransformer):
    """Load a small Hugging Face model: cached copy first (no network), download only if missing."""
    try:
        return cls(name, local_files_only=True)
    except Exception:
        try:
            return cls(name)                                 # first run only: ~90 MB download
        except Exception as e:
            raise RuntimeError(f"Could not load {name}: connect to the internet once so it can download.") from e


embedder = load_st("sentence-transformers/all-MiniLM-L6-v2")


def encode(texts):
    """Unit-length MiniLM embeddings (384-d), so a dot product IS the cosine similarity."""
    return embedder.encode(list(texts), normalize_embeddings=True, convert_to_numpy=True)


print("embedder: all-MiniLM-L6-v2 |", embedder.get_embedding_dimension(), "dims | max",
      embedder.max_seq_length, "tokens per text (the rest is silently cut off)")

## What needs a real model, and what runs offline

Every technique below is implemented the way you would ship it, calling the LLM through `client`. Offline, the stand-in model returns **canned** rewrites (it cannot paraphrase), so each LLM technique is paired with a **deterministic** variant (a domain glossary, a rule-based splitter, a hand-written example clearly labelled as such) so the mechanics and the charts still mean something without a key.

| Technique | LLM version (needs a real model to shine) | Deterministic version used offline |
|---|---|---|
| Query rewriting / expansion | LLM rewrites into documentation vocabulary | domain glossary (synonyms, acronyms) - a real production technique too |
| Multi-query + RRF | LLM writes 3 paraphrases | original + glossary-expanded + keywords-only |
| HyDE | LLM writes a hypothetical answer | one hand-written hypothetical answer (labelled) |
| Step-back | LLM writes a broader question | one hand-written step-back question (labelled) |
| Decomposition | LLM splits into sub-questions | split on "..., and how/what ..." |
| Contextual compression | LLM extracts relevant sentences | embedding similarity per sentence |
| Self-query filters | LLM extracts a filter from the question | session metadata (who is asking) |
| Agentic loop | LLM judges sufficiency and writes the follow-up | same loop, with the judge's follow-up checked by hand |

## The baseline and the eval harness

Baseline = the **dense lane alone** (MiniLM cosine over section chunks), so each technique's effect is visible. In production you stack these on top of hybrid retrieval and reranking. Metrics on the 14 answerable questions: **hit@1**, **hit@3**, and **all@3** (every piece of evidence is in the top 3 - what multi-part and multi-hop questions need).

In [ ]:
chunk_vecs = encode([c["text"] for c in CHUNKS])


def dense_search(query, k=10):
    return list(np.argsort(-(chunk_vecs @ encode([query])[0]), kind="stable")[:k])


def rrf(ranked_lists, k=60):
    scores = {}
    for lst in ranked_lists:
        for rank, doc in enumerate(lst, start=1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=lambda d: -scores[d])


def covers(ids, q):
    """How many of the question's evidence items appear in these chunks."""
    return sum(any(CHUNKS[i]["doc"] == d and p.lower() in CHUNKS[i]["text"].lower() for i in ids) for d, p in q["evidence"])


def evaluate(search_fn, questions=IN_SCOPE):
    """search_fn(question_text) -> ranked chunk ids."""
    rows = []
    for q in questions:
        ids = search_fn(q["q"])
        rows.append({"hit@1": covers(ids[:1], q) > 0, "hit@3": covers(ids[:3], q) > 0,
                     "all@3": covers(ids[:3], q) == len(q["evidence"])})
    return pd.DataFrame(rows).mean()


RESULTS = {"baseline (dense)": evaluate(dense_search)}
print(RESULTS["baseline (dense)"].round(3).to_string())
HARD = [q for q in IN_SCOPE if covers(dense_search(q["q"])[:1], q) == 0]
print("\nquestions the baseline gets wrong at rank 1 (used for the LLM variants, to keep calls few):")
for q in HARD:
    print(f"  [{q['kind']}] {q['q']}")

## 1. Query rewriting and expansion

**Failure it fixes:** vocabulary mismatch - the user says "money back", "parcel", "down more than promised"; the documents say "refund", "shipment", "uptime commitment" and "service credit".

### 1a. Deterministic: a domain glossary

A hand-maintained synonym/acronym map appended to the query (Elasticsearch calls this a synonym filter). Cheap, predictable, and it only covers what you thought of.

In [ ]:
GLOSSARY = {
    "money back": "refund", "parcel": "shipment delivery", "showed up": "arrived",
    "vacation": "paid time off", "holiday": "paid time off",
    "compensated": "service credit", "outage": "uptime service credit", "down": "uptime outage service credit",
    "get back": "credit refund", "promised": "commitment SLA",
    "outdoors": "outdoor rugged IP67", "sso": "single sign-on SAML",
}


def glossary_rewrite(query):
    extra = [v for k, v in GLOSSARY.items() if re.search(r"\b" + re.escape(k), query.lower())]
    return query + (" | " + " ".join(extra) if extra else "")


vague = next(q for q in IN_SCOPE if q["kind"] == "vague")
print(glossary_rewrite(vague["q"]))
RESULTS["glossary rewrite"] = evaluate(lambda s: dense_search(glossary_rewrite(s)))
print(pd.DataFrame(RESULTS).T.round(3))
assert covers(dense_search(glossary_rewrite(vague["q"]))[:3], vague) and not covers(dense_search(vague["q"])[:3], vague)

The vague SLA question - missed by every lane in notebooks 04 and 05 - is fixed by adding the documentation's own words. Exact identifiers (`ERR-429-A`) are *not* a vocabulary problem; that one needs the BM25 lane, not a rewrite.

### 1b. LLM rewriting

In [ ]:
REWRITE_SYSTEM = ("Rewrite the user's question as a short search query for Acme Cloud's documentation. "
                  "Use the vocabulary a policy or product document would use. Return only the query.")


def llm_rewrite(query):
    resp = client.chat.completions.create(model=MODEL, temperature=0, max_tokens=60,
                                          messages=[{"role": "system", "content": REWRITE_SYSTEM},
                                                    {"role": "user", "content": query}])
    return resp.choices[0].message.content.strip()


llm_rewrites = {q["q"]: llm_rewrite(q["q"]) for q in HARD}                    # len(HARD) LLM calls
for original, rewritten in llm_rewrites.items():
    print(f"{original}\n   -> {rewritten[:110]}")

Offline, the "rewrite" is the stand-in's canned placeholder text: it still contains the original question, but wrapped in boilerplate, so offline it does *worse* than the baseline below - an honest reminder to evaluate every rewrite before shipping it. A real model returns something like *"service credit compensation when monthly uptime falls below the SLA commitment"* for the vague question - the glossary's effect, without maintaining a glossary. Evaluate on the hard questions with whatever provider you have:

In [ ]:
def llm_rewrite_search(s):
    return dense_search(llm_rewrites.get(s, s))


hard_table = pd.DataFrame({"baseline": evaluate(dense_search, HARD),
                           "glossary rewrite": evaluate(lambda s: dense_search(glossary_rewrite(s)), HARD),
                           f"LLM rewrite ({PROVIDER})": evaluate(llm_rewrite_search, HARD)}).T
print(f"on the {len(HARD)} hard questions:\n", hard_table.round(3))

## 2. Multi-query expansion + RRF

**Failure it fixes:** the recall ceiling of one phrasing. Search with several phrasings and fuse the ranked lists with RRF (the same function as notebook 04 - RRF fuses *any* ranked lists). The searches run in parallel, so latency ≈ one search + the rewrite call.

In [ ]:
STOPWORDS = set("""a an the is are was were be been of to in on for with and or but if then so as at by from this that
it its i you we my our your do does did can how what who when which why there their they me get more than""".split())


def keywords_only(query):
    return " ".join(w for w in re.findall(r"[a-z0-9-]+", query.lower()) if w not in STOPWORDS)


def deterministic_variants(query):
    return list(dict.fromkeys([query, glossary_rewrite(query), keywords_only(query)]))


def multi_query_search(query, variants_fn=deterministic_variants):
    return rrf([dense_search(v) for v in variants_fn(query)])


print(deterministic_variants(vague["q"]))
RESULTS["multi-query (3 variants) + RRF"] = evaluate(multi_query_search)
print(pd.DataFrame(RESULTS).T.round(3))

In [ ]:
class Queries(BaseModel):
    queries: list[str] = Field(description="3 different search queries for the same information need")


MULTI_SYSTEM = ("Write 3 different search queries that would find documentation answering the user's question. "
                "Vary the vocabulary: synonyms, the terms a policy document would use, key entities.")


def llm_variants(query):
    resp = client.chat.completions.parse(model=MODEL, temperature=0, response_format=Queries,
                                         messages=[{"role": "system", "content": MULTI_SYSTEM},
                                                   {"role": "user", "content": query}])
    generated = [g for g in resp.choices[0].message.parsed.queries if g.strip()]
    return list(dict.fromkeys([query] + generated))                         # always keep the original


llm_mq = {q["q"]: llm_variants(q["q"]) for q in HARD}                        # len(HARD) LLM calls
print(f"variants for the vague question ({PROVIDER}):", llm_mq[vague["q"]])
print(pd.DataFrame({"baseline": evaluate(dense_search, HARD),
                    f"LLM multi-query ({PROVIDER})": evaluate(lambda s: rrf([dense_search(v) for v in llm_mq[s]]), HARD)}).T.round(3))

Offline, structured output returns a schema instance filled mechanically from the prompt (a fragment of the question), so the "fan-out" is degenerate and its numbers here mean nothing; the code always keeps the original query, so nothing breaks. A real model returns three genuinely different phrasings, e.g. *"SLA service credits for downtime"*, *"compensation when uptime is below 99.9%"*, *"what happens if the platform misses its uptime commitment"*.

**When it pays:** multi-aspect or vague questions, or when the first retrieval's scores are flat. **Cost:** 1 extra LLM call + N searches per question - do it conditionally.

## 3. HyDE - search with a hypothetical answer

**Failure it fixes:** question/passage asymmetry. A short question and a factual passage look different even when they are about the same thing; a *hypothetical answer* looks like the passage. Generate one (it may contain wrong details - that is fine, only its embedding is used), embed it, search with it.

First the mechanism, with a hand-written hypothetical answer of the kind an LLM writes (**labelled: written by hand for this demo**):

In [ ]:
HANDWRITTEN_HYDE = ("When the service is down for longer than the SLA allows, customers are compensated with a credit "
                    "on their bill, usually a percentage of that month's fee.")
evidence_idx = [i for i, c in enumerate(CHUNKS) if is_relevant(c, vague)]
q_vec, h_vec = encode([vague["q"]])[0], encode([HANDWRITTEN_HYDE])[0]
for name, v in [("question", q_vec), ("hypothetical answer", h_vec)]:
    order = list(np.argsort(-(chunk_vecs @ v)))
    rank = min(order.index(i) for i in evidence_idx) + 1
    print(f"search with the {name:19s}: cosine to the evidence chunk = {float(chunk_vecs[evidence_idx[0]] @ v):.3f}, rank = {rank}")
assert float(chunk_vecs[evidence_idx[0]] @ h_vec) > float(chunk_vecs[evidence_idx[0]] @ q_vec)

sim_q, sim_h = chunk_vecs @ q_vec, chunk_vecs @ h_vec
plt.figure(figsize=(6.5, 4.2))
plt.scatter(sim_q, sim_h, color="#adb5bd", label="other chunks")
plt.scatter(sim_q[evidence_idx], sim_h[evidence_idx], color="#e8590c", s=90, label="evidence chunk (SLA > Service credits)")
lim = [min(sim_q.min(), sim_h.min()) - 0.05, max(sim_q.max(), sim_h.max()) + 0.05]
plt.plot(lim, lim, "k:", lw=1, label="no change")
plt.xlabel("cosine to the QUESTION")
plt.ylabel("cosine to the HYPOTHETICAL ANSWER")
plt.title("HyDE: the answer-shaped query lifts the evidence chunk")
plt.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
HYDE_SYSTEM = ("Write a short passage (2-3 sentences) from Acme Cloud's documentation that answers the question. "
               "If you do not know the details, write a plausible passage anyway.")


def hyde_passage(query):
    resp = client.chat.completions.create(model=MODEL, temperature=0, max_tokens=120,
                                          messages=[{"role": "system", "content": HYDE_SYSTEM},
                                                    {"role": "user", "content": query}])
    return resp.choices[0].message.content


hyde_docs = {q["q"]: hyde_passage(q["q"]) for q in HARD}                      # len(HARD) LLM calls
print(f"HyDE passage for the vague question ({PROVIDER}):\n  {hyde_docs[vague['q']][:160]}")
print(pd.DataFrame({"baseline": evaluate(dense_search, HARD),
                    f"HyDE ({PROVIDER})": evaluate(lambda s: dense_search(hyde_docs[s]), HARD),
                    "HyDE + question, RRF": evaluate(lambda s: rrf([dense_search(s), dense_search(hyde_docs[s])]), HARD)}).T.round(3))

With the stand-in, the "hypothetical passage" is canned text, so HyDE cannot help offline; with a real model it behaves like the hand-written example. Two cautions: HyDE can **hurt** when the invented answer drifts off-topic (fusing it with the original query, as in the last row, limits the damage), and an **asymmetric embedding model** trained with query/passage prefixes (E5, BGE) solves the same asymmetry with no extra call - try that first.

## 4. Step-back prompting

**Failure it fixes:** questions too specific or oddly phrased to match anything, where the *background* section would help. Ask a broader question too, retrieve for both, fuse. Hand-written step-back question (**labelled**), then the LLM version:

In [ ]:
HANDWRITTEN_STEPBACK = "What does Acme's service level agreement promise customers?"
fused = rrf([dense_search(vague["q"]), dense_search(HANDWRITTEN_STEPBACK)])
print("step-back question:", HANDWRITTEN_STEPBACK)
print("evidence rank - original only:", next((r for r, i in enumerate(dense_search(vague["q"], 42), 1) if i in evidence_idx), None),
      "| original + step-back (RRF):", next((r for r, i in enumerate(fused, 1) if i in evidence_idx), None))

STEPBACK_SYSTEM = "Rewrite the question as a broader, more general 'step-back' question about the underlying topic. Return only the question."
resp = client.chat.completions.create(model=MODEL, temperature=0, max_tokens=60,
                                      messages=[{"role": "system", "content": STEPBACK_SYSTEM},
                                                {"role": "user", "content": vague["q"]}])
print(f"LLM step-back ({PROVIDER}):", resp.choices[0].message.content[:120])

The background question pulls the SLA sections up and the fused ranking improves - not necessarily to the top, which is why step-back is usually combined with a reranker. A real model writes a question much like the hand-written one; the stand-in's reply is canned.

## 5. Query decomposition (multi-part and multi-hop)

**Failure it fixes:** one embedding of a two-part question sits between the two meanings and is near neither. Split it, retrieve for each part, merge.

In [ ]:
def rule_decompose(query):
    parts = re.split(r",?\s+and\s+(?=(?:how|what|which|who|when|where|is|are|can|do|does)\b)", query, flags=re.I)
    return [p.strip(" ?") + "?" for p in parts if p.strip()]


def decomposed_search(query, per_part=2):
    parts = rule_decompose(query)
    if len(parts) == 1:
        return dense_search(query)
    picked = [i for p in parts for i in dense_search(p)[:per_part]]           # the best chunks of EACH part first
    return list(dict.fromkeys(picked + rrf([dense_search(p) for p in parts])))


multi_part = next(q for q in IN_SCOPE if q["kind"] == "multi-part")
print(rule_decompose(multi_part["q"]))
print("evidence found in top 3 - baseline:", covers(dense_search(multi_part["q"])[:3], multi_part), "of 2 |",
      "decomposed:", covers(decomposed_search(multi_part["q"])[:3], multi_part), "of 2")
RESULTS["decomposition (rule)"] = evaluate(decomposed_search)
assert covers(decomposed_search(multi_part["q"])[:3], multi_part) == 2 > covers(dense_search(multi_part["q"])[:3], multi_part)

In [ ]:
class SubQuestions(BaseModel):
    sub_questions: list[str] = Field(description="self-contained sub-questions, in the order they must be answered")


multi_hop = next(q for q in IN_SCOPE if q["kind"] == "multi-hop")
resp = client.chat.completions.parse(
    model=MODEL, temperature=0, response_format=SubQuestions,
    messages=[{"role": "system", "content": "Split the user's question into the sub-questions needed to answer it. "
                                            "If one step depends on another, list the dependency first."},
              {"role": "user", "content": multi_hop["q"]}])
print(f"LLM decomposition ({PROVIDER}):", resp.choices[0].message.parsed.sub_questions)

A real model decomposes the multi-hop question into *"Which is the cheapest Acme plan that includes SSO?"* then *"What is the API rate limit on that plan?"* - the second depends on the answer to the first, so the parts must run **in sequence** (that is what makes it multi-hop rather than multi-part). The stand-in returns the question unchanged. Sequential hops are what the agentic loop in section 9 automates.

## 6. Contextual compression

**Failure it fixes:** context bloat - retrieved chunks carry sentences unrelated to the question, costing tokens and diluting attention. Keep only the sentences that match the question (an embedding filter here; LangChain's `LLMChainExtractor` does it with an LLM call per chunk).

In [ ]:
def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", text) if s.strip()]


def compress(query, ids, keep=3):
    sents = [(i, s) for i in ids for s in split_sentences(CHUNKS[i]["text"].split("\n", 1)[1])]
    sims = encode([s for _, s in sents]) @ encode([query])[0]
    best = sorted(np.argsort(-sims)[:keep])                                   # top sentences, in document order
    return [sents[j] for j in best]


rows = []
for q in IN_SCOPE:
    ids = dense_search(q["q"])[:3]
    full = " ".join(CHUNKS[i]["text"] for i in ids)
    kept = compress(q["q"], ids)
    kept_text = " ".join(s for _, s in kept)
    rows.append({"chars before": len(full), "chars after": len(kept_text),
                 "evidence kept": any(p.lower() in kept_text.lower() for _, p in q["evidence"]) or not covers(ids, q),
                 "evidence was there": covers(ids, q) > 0})
comp = pd.DataFrame(rows)
print(f"context size: {comp['chars before'].mean():.0f} -> {comp['chars after'].mean():.0f} characters per question "
      f"({1 - comp['chars after'].sum() / comp['chars before'].sum():.0%} smaller)")
kept_rate = comp.loc[comp["evidence was there"], "evidence kept"].mean()
print(f"evidence still present after compression (when it was retrieved): {kept_rate:.0%}")
assert comp["chars after"].sum() < 0.8 * comp["chars before"].sum()

A large cut in tokens, at the price of occasionally dropping the evidence sentence - which is why compression is tuned (how many sentences to keep) against the same eval set, and why an LLM extractor, which understands the question, usually does better than a similarity filter.

## 7. Parent-document retrieval (small-to-big)

Index single sentences for precise matching; return the whole section they belong to, deduplicated (the pattern from [notebook 01](01_chunking_strategies_and_semantic_chunking.ipynb), now measured):

In [ ]:
children = [(pid, s) for pid, c in enumerate(CHUNKS) for s in split_sentences(c["text"])]
child_vecs = encode([s for _, s in children])


def parent_search(query, k_children=10):
    order = np.argsort(-(child_vecs @ encode([query])[0]))[:k_children]
    return list(dict.fromkeys(children[j][0] for j in order))                 # parents, deduplicated, rank order


RESULTS["parent-document (sentence -> section)"] = evaluate(parent_search)
print(pd.DataFrame(RESULTS).T.round(3))

## 8. Metadata filtering and self-query

**Failure it fixes:** the right answer is outranked by content from the wrong *scope* - another product, an old version, another tenant, or (here) the employee handbook answering a customer. Filter **before** ranking.

Two very different sources of a filter:

- **From the session** - who is asking (tenant, role, permissions). Never from the LLM or the user's text: a filter the user can influence is not a security control.
- **From the question** ("self-query") - an LLM extracts content filters like product or topic.

In [ ]:
EMPLOYEE_DOCS = {"pto_policy.md", "oncall_policy.md", "expenses.md"}
AUDIENCE = {c["doc"]: ("employee" if c["doc"] in EMPLOYEE_DOCS else "customer") for c in CHUNKS}


def filtered_search(query, audience=None, k=10):
    allowed = [i for i, c in enumerate(CHUNKS) if audience is None or AUDIENCE[c["doc"]] == audience]
    sims = chunk_vecs[allowed] @ encode([query])[0]
    return [allowed[j] for j in np.argsort(-sims, kind="stable")[:k]]         # pre-filter, then rank


comp_q = next(q for q in IN_SCOPE if q["q"].startswith("Do we get compensated"))
print(comp_q["q"])
print("  no filter          :", [CHUNKS[i]["id"] for i in dense_search(comp_q["q"])[:3]])
print("  audience = customer:", [CHUNKS[i]["id"] for i in filtered_search(comp_q["q"], "customer")[:3]])
assert covers(filtered_search(comp_q["q"], "customer")[:1], comp_q) == 1 > covers(dense_search(comp_q["q"])[:1], comp_q)

Unfiltered, the on-call *compensation* section (an employee document) wins on the word "compensated"; filtered to what a customer may see, the SLA's service credits come first. Now the self-query part - the LLM proposes a content filter, and the code validates it:

In [ ]:
TOPIC = {"api_errors.md": "api", "api_rate_limits.md": "api", "webhooks.md": "api", "gateway_models.md": "hardware",
         "shipping.md": "hardware", "warranty.md": "hardware", "pricing.md": "billing", "refund_policy.md": "billing",
         "sla.md": "billing", "sso_setup.md": "security", "account_security.md": "security", "data_retention.md": "security",
         "pto_policy.md": "hr", "oncall_policy.md": "hr", "expenses.md": "hr"}


class SearchFilter(BaseModel):
    topic: Literal["billing", "hardware", "api", "security", "hr"]
    query: str = Field(description="the question without the filter words")


question = "How many sensors does the EG-300 support?"
resp = client.chat.completions.parse(
    model=MODEL, temperature=0, response_format=SearchFilter,
    messages=[{"role": "system", "content": "Extract a search filter. Topics: billing, hardware, api, security, hr."},
              {"role": "user", "content": question}])
flt = resp.choices[0].message.parsed
allowed = [i for i, c in enumerate(CHUNKS) if TOPIC[c["doc"]] == flt.topic]
hits = [allowed[j] for j in np.argsort(-(chunk_vecs[allowed] @ encode([question])[0]))[:3]]
if not any(is_relevant(CHUNKS[i], IN_SCOPE[3]) for i in hits):
    print(f"filter topic={flt.topic!r} ({PROVIDER}) -> top hits {[CHUNKS[i]['id'] for i in hits]}: the filter is wrong, "
          "fall back to unfiltered search")
else:
    print(f"filter topic={flt.topic!r} ({PROVIDER}) -> top hits {[CHUNKS[i]['id'] for i in hits]}")

A real model returns `topic="hardware"` for the EG-300 question. The offline stand-in fills the enum with the first option it can justify from the text (here none, so the first one), which lands in the wrong topic - a useful reminder that **LLM-extracted filters must be validated** (does the filter return anything relevant? is it a known value?) with a fallback to unfiltered search, because a wrong pre-filter silently removes the answer.

## 9. Agentic RAG: retrieve -> judge -> re-query (with a budget)

**Failure it fixes:** multi-hop questions and bad first searches. The model looks at what it retrieved, decides whether it is enough, and if not writes the next query. It must have a **step budget** and a trace.

In [ ]:
class Verdict(BaseModel):
    enough: bool = Field(description="true if the passages contain everything needed to answer")
    follow_up_query: str = Field(description="if not enough: the next search query; else empty")


JUDGE_SYSTEM = ("You check retrieved passages against a question. Decide if they contain everything needed to answer. "
                "If they do not, write the single search query that would find what is missing.")


def llm_judge(question, ids):
    passages = "\n".join(f"[{CHUNKS[i]['id']}] {CHUNKS[i]['text'].replace(chr(10), ' ')}" for i in ids)
    resp = client.chat.completions.parse(model=MODEL, temperature=0, response_format=Verdict,
                                         messages=[{"role": "system", "content": JUDGE_SYSTEM},
                                                   {"role": "user", "content": f"Question: {question}\n\nPassages:\n{passages}"}])
    return resp.choices[0].message.parsed


def agentic_retrieve(question, judge=llm_judge, max_steps=3, per_step=2):
    gathered, query, trace = [], question, []
    for step in range(1, max_steps + 1):
        new = [i for i in dense_search(query) if i not in gathered][:per_step]
        gathered += new
        verdict = judge(question, gathered)
        trace.append({"step": step, "query": query[:60], "retrieved": [CHUNKS[i]["id"] for i in new],
                      "enough": verdict.enough, "follow-up": verdict.follow_up_query[:50]})
        if verdict.enough or not verdict.follow_up_query.strip():
            break
        query = verdict.follow_up_query
    return gathered, pd.DataFrame(trace)


ids, trace = agentic_retrieve(multi_hop["q"])                                # <= 3 LLM calls
print(f"provider: {PROVIDER}")
print(trace.to_string(index=False))
print("evidence covered:", covers(ids, multi_hop), "of", len(multi_hop["evidence"]))
assert len(trace) <= 3, "the step budget must hold whatever the judge says"

With a real model the trace typically reads: step 1 retrieves the rate-limit table ("Business: 1,200 requests per minute ...") but nothing says which plan includes SSO → `enough=false`, follow-up *"Which Acme plan includes single sign-on?"* → step 2 retrieves the pricing section → `enough=true`. Offline, the stand-in's verdicts are canned schema instances, so the loop just runs until the budget stops it - which is exactly the guard you need in production.

We cannot fake the judgement offline, but we can check the retrieval step a real judge would trigger:

In [ ]:
follow_up = "Which Acme plan includes single sign-on?"          # what a real judge writes after step 1
step1 = dense_search(multi_hop["q"])[:2]
step2 = [i for i in dense_search(follow_up) if i not in step1][:2]
print("step 1:", [CHUNKS[i]["id"] for i in step1], "-> evidence", covers(step1, multi_hop), "of 2")
print("step 2:", [CHUNKS[i]["id"] for i in step2], "-> evidence", covers(step1 + step2, multi_hop), "of 2")
assert covers(step1 + step2, multi_hop) == 2 > covers(step1, multi_hop)

| | Classic RAG | Agentic RAG |
|---|---|---|
| Retrievals | exactly one | zero to many, model decides |
| Recovers from a bad search | no | yes - judge and re-query |
| Multi-hop | poorly | well, hop by hop |
| Latency / cost | predictable: 1 LLM call | variable: 2-10 LLM calls |
| Needs | nothing extra | a step budget, a cost ceiling, tracing |
| Use when | FAQ-style lookups (most traffic) | multi-hop, ambiguous, multi-source questions |

## 10. GraphRAG, briefly

**Failure it fixes:** questions about *relationships* or *aggregates* that no single chunk answers ("cheapest plan with SSO", "which themes recur across 900 incident reports"). GraphRAG extracts entities and relations into a graph at index time (with an LLM over the whole corpus - expensive) and answers by traversal. A toy version, with the relations pulled from the two relevant sections by regex:

In [ ]:
pricing_text = next(c["text"] for c in CHUNKS if c["id"] == "pricing.md#0")
limits_text = next(c["text"] for c in CHUNKS if c["id"] == "api_rate_limits.md#0")
graph = {}                                                            # plan -> {relation: value}
for plan, price in re.findall(r"(Free|Pro|Business) costs \$(\d+)", pricing_text):
    graph.setdefault(plan, {})["price_per_user"] = int(price)
graph.setdefault("Enterprise", {})["price_per_user"] = float("inf")   # "custom" pricing
for plan, limit in re.findall(r"(Free|Pro|Business|Enterprise): ([\d,]+) requests per minute", limits_text):
    graph.setdefault(plan, {})["rate_limit_per_min"] = int(limit.replace(",", ""))
for plan in ("Business", "Enterprise"):                              # "SSO is included from the Business plan upward"
    graph[plan]["includes_sso"] = True
print(graph)

with_sso = [p for p, attrs in graph.items() if attrs.get("includes_sso")]
cheapest = min(with_sso, key=lambda p: graph[p]["price_per_user"])
print(f"cheapest plan with SSO: {cheapest} -> rate limit {graph[cheapest]['rate_limit_per_min']:,} requests/minute")
assert cheapest == "Business" and graph[cheapest]["rate_limit_per_min"] == 1200

The traversal answers exactly, with no similarity search - once the graph exists. Building and *maintaining* it (re-extracting when documents change, designing the schema) is the cost, so GraphRAG is worth it when users ask relational or global questions, and overkill for lookups. Most systems that need it keep vector RAG alongside.

## 11. All techniques on the same eval set

In [ ]:
summary = pd.DataFrame(RESULTS).T
print(summary.round(3))
ax = summary[["hit@1", "hit@3", "all@3"]].plot(kind="barh", figsize=(10, 4.2), color=["#74c0fc", "#3b5bdb", "#e8590c"])
ax.set_xlim(0, 1.05)
ax.invert_yaxis()
ax.set_xlabel(f"fraction of {len(IN_SCOPE)} answerable questions (dense lane, deterministic variants)")
ax.set_title("Which query-side technique moves which metric")
plt.tight_layout()
plt.show()

Read it per metric, not as a leaderboard: the glossary and multi-query mostly move **hit@1 / hit@3** (vocabulary), decomposition moves **all@3** (multi-part), and parent-document retrieval lifts hit@3 because one precise sentence match brings its whole section along. On 14 questions one question is 7 percentage points, so treat small differences as noise.

## Which technique for which failure

| Failure you observe | Diagnose with | Technique | Cost |
|---|---|---|---|
| User words ≠ document words ("money back" vs "refund") | evidence ranks low for paraphrases | query rewriting / glossary expansion, multi-query | 0-1 LLM call |
| Follow-up question ("what about the second one?") | query has pronouns, retrieves nothing | contextualise the query with the chat history | 1 LLM call |
| Short question vs long passages; new domain | good docs rank mid-list | HyDE (or an asymmetric embedder) | 1 LLM call |
| Too specific, needs background | nothing matches the exact phrasing | step-back + fuse | 1 LLM call |
| Two questions in one | only one part's evidence retrieved | decomposition, retrieve per part | 0-1 LLM call |
| Needs the answer to one lookup to make the next | second hop never retrieved | iterative / agentic retrieval with a step budget | 2-10 LLM calls |
| Right text, wrong scope (tenant, version, audience) | wrong-scope chunks outrank | metadata pre-filtering (session), self-query (content) | ~0 |
| Chunks too long or noisy | context tokens high, answers diluted | contextual compression, rerank, smaller k | 0-1 call per chunk |
| Precise match, incomplete context | answers miss the surrounding rules | parent-document retrieval | ~0 (two stores) |
| Relationships / aggregates across documents | no single chunk contains the answer | GraphRAG, or text-to-SQL for tables | large index-time cost |
| Exact identifiers missed | codes like `ERR-429-A` rank low | **not a rewrite problem** - add the BM25 lane (notebook 04) | ~0 |

## Try it yourself

**Exercise 1.** Conditional rewriting: only apply the glossary rewrite when the baseline's top cosine is below 0.5 (a "retrieval looks weak" signal). How many questions get rewritten, and does quality hold?

In [ ]:
# Solution — try it yourself first, then run this
def conditional_search(s, floor=0.5):
    weak = float((chunk_vecs @ encode([s])[0]).max()) < floor
    return dense_search(glossary_rewrite(s) if weak else s)


n_rewritten = sum(float((chunk_vecs @ encode([q["q"]])[0]).max()) < 0.5 for q in IN_SCOPE)
print(f"rewritten: {n_rewritten} of {len(IN_SCOPE)} questions")
print(pd.DataFrame({"always rewrite": evaluate(lambda s: dense_search(glossary_rewrite(s))),
                    "rewrite only when weak": evaluate(conditional_search)}).T.round(3))

**Exercise 2.** Stack techniques: multi-query variants **and** decomposition (decompose first, then generate variants for each part), fused with RRF. Does all@3 improve on the multi-part question without hurting the rest?

In [ ]:
# Solution — try it yourself first, then run this
def stacked_search(s):
    lists = [dense_search(v) for part in rule_decompose(s) for v in deterministic_variants(part)]
    parts = rule_decompose(s)
    head = [i for p in parts for i in multi_query_search(p)[:2]] if len(parts) > 1 else []
    return list(dict.fromkeys(head + rrf(lists)))


print(pd.DataFrame({"multi-query": RESULTS["multi-query (3 variants) + RRF"],
                    "decomposition": RESULTS["decomposition (rule)"],
                    "decomposition + multi-query": evaluate(stacked_search)}).T.round(3))

**Exercise 3.** Extend the toy graph: add a `retention_months` relation for each plan from the data-retention section, and answer "How long is event data kept on the cheapest plan with SSO?" by traversal.

In [ ]:
# Solution — try it yourself first, then run this
retention_text = next(c["text"] for c in CHUNKS if c["id"] == "data_retention.md#0")
for months, plans in re.findall(r"(\d+) months on ([A-Za-z ]+?)(?:,|\.)", retention_text):
    for plan in re.split(r"\s+and\s+", plans.strip()):
        graph.setdefault(plan, {})["retention_months"] = int(months)
print({p: a.get("retention_months") for p, a in graph.items()})
print(f"{cheapest}: event data kept {graph[cheapest]['retention_months']} months")

## Say it in an interview

- **30 seconds:** "First I find out *why* retrieval fails on the eval set. Vocabulary mismatch: rewrite or expand the query, or fan out into 3-5 phrasings and fuse with RRF. Question/passage asymmetry: HyDE, or an asymmetric embedder. Multi-part: decompose and retrieve per part. Multi-hop: iterative retrieval, where an agent judges whether it has enough and writes the next query, under a hard step budget. Wrong scope: metadata filters before ranking - from the session for permissions, from the question for content. Each costs an LLM call, so I apply them conditionally, when first-pass retrieval looks weak."
- **The ordering (`rg15`):** measure recall → fix chunking → add BM25 (hybrid) → metadata filters → reranker → query rewriting → contextual retrieval → better / fine-tuned embedder.
- **Numbers:** multi-query 3-5 variants; agent loops 2-3 hops; every rewrite ≈ one small-model call (~100-500 ms).
- **Traps:** rewriting every query (doubles latency for the majority that did not need it); HyDE drifting off-topic; trusting LLM-extracted filters; letting the user's text influence a permission filter; an agent loop without a budget; reaching for GraphRAG for simple lookups.
- **Follow-ups:** "how do you know a rewrite helped?" (same frozen eval set, per failure class - aggregates hide regressions); "multi-turn RAG?" (contextualise the query against the chat history before retrieval - the most common real-world rewrite).

## Next

- [RAG evaluation: retrieval and faithfulness](../12_evaluation/03_rag_evaluation_retrieval_and_faithfulness.ipynb) - measure these changes properly.
- [Agent loop from scratch](../15_agentic_ai/01_agent_loop_from_scratch.ipynb) · [ReAct and plan-and-execute](../15_agentic_ai/03_react_and_plan_and_execute.ipynb) · [LangGraph conditional edges and loops](../14_langgraph/02_conditional_edges_and_loops.ipynb)
- [Hybrid search and RRF](04_hybrid_search_and_rrf.ipynb) · [Reranking](05_reranking.ipynb) · [RAG pipeline from scratch](06_rag_pipeline_from_scratch.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) (agentic vs classic RAG side by side) · [AI system design concepts](../../ai_system_design_concepts/index.html) · [agentic AI flow](../../agentic_ai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)